# 08 - Deploy the Workspace Owner Agent

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Creates or updates and publishes a **separate owner Agent**, grounded only on the secured Workspace Owner semantic model. It does not modify the model, grant permissions or change the central Agent or app.

**Before running:** configure the owner model's fixed-identity connection with source SSO disabled, run an owner-enabled review through Gold, then complete **07_OwnerAccessSync**. Setup stamps the exact owner model ID below. Do not substitute the governance model.

**Run order:** run the SDK install cell first. If Fabric restarts the kernel, then run the parameters and deployment cells below. This standalone notebook is not a pipeline activity.

Model RLS and each caller's permissions enforce workspace access. Instructions and table selection are not security boundaries. After publication, approve only controlled test readers: query-only Agent access, owner-model Read and WorkspaceOwner membership. Build, FAR workspace roles and raw Lakehouse/SQL access are not required. Test actual users, denied workspaces, expiry and revocation before broader sharing. Maintain daily 07 sync.

See the [owner Agent guide](https://github.com/microsoft/fabric-architecture-review/blob/main/docs/workspace-owner-agent.md).


In [ ]:
%pip install -q fabric-data-agent-sdk==0.1.30a0


In [ ]:
GITHUB_REPO_URL = "https://github.com/microsoft/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""
WORKSPACE_ID = ""
OWNER_SEMANTIC_MODEL_ID = ""
OWNER_AGENT_NAME = "Fabric Arch Review - Workspace Owner Agent"


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path
from uuid import UUID

UUID(WORKSPACE_ID)
UUID(OWNER_SEMANTIC_MODEL_ID)
if not isinstance(OWNER_AGENT_NAME, str) or not OWNER_AGENT_NAME.strip():
    raise ValueError("OWNER_AGENT_NAME must be a nonempty string.")

WORK_ROOT = "/tmp/fabric-arch-review-owner-agent"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR], check=True)
del _url
if REPO_DIR in sys.path:
    sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
import importlib
for _module_name in list(sys.modules):
    if _module_name.split(".", 1)[0] in ("collectors", "analyzers", "reports", "orchestration"):
        sys.modules.pop(_module_name, None)
importlib.invalidate_caches()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

from orchestration.fabric_api import FabricClient
from reports.agent.owner_agent import deploy_owner_agent

_version = Path(REPO_DIR, "VERSION").read_text(encoding="utf-8-sig").strip()
_agent_id = deploy_owner_agent(
    client=FabricClient(lambda: notebookutils.credentials.getToken("pbi")),
    workspace_id=WORKSPACE_ID, model_id=OWNER_SEMANTIC_MODEL_ID,
    agent_name=OWNER_AGENT_NAME, version=_version,
)
from orchestration.folders import organize
_folder_client = FabricClient(lambda: notebookutils.credentials.getToken('pbi'))
organize(_folder_client, WORKSPACE_ID, [(_agent_id, 'DataAgent', 'Agents')])
_ctx = notebookutils.runtime.context
organize(_folder_client, _ctx.get('currentWorkspaceId') or _ctx.get('workspaceId'),
         [(_ctx['currentNotebookId'], 'Notebook', 'Notebooks')])
print("Workspace Owner Agent published and source verified:", _agent_id)
print("No permissions were granted. Complete actual-reader acceptance before broader sharing; keep daily 07 access sync active.")
